# LA Studio translation — Tencent Hy-MT2 1.8B

This notebook loads exactly `hy-mt2-1.8b` (`tencent/Hy-MT2-1.8B`) on CUDA.
It is independent from API Gateway and rejects every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into the matching LA Studio feature.


In [ ]:
!nvidia-smi
%pip install -q "fastapi==0.115.12" "uvicorn==0.34.3" "transformers>=5.6.0,<6" "accelerate>=1.12,<2" "sentencepiece==0.2.1" "safetensors>=0.6,<1"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_translation_worker.py')
WORKER.write_text('import os\nimport secrets\nimport threading\n\nimport torch\nfrom fastapi import Depends, FastAPI, Header, HTTPException\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.")\n\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nMODEL_ID = "hy-mt2-1.8b"\nMODEL_NAME = "Tencent Hy-MT2 1.8B"\nUPSTREAM_MODEL = "tencent/Hy-MT2-1.8B"\nUPSTREAM_REVISION = "9a341cd1b679d3efd23b46e847b01745a71ed792"\nLANGUAGE_NAMES = {\n    "zh": "Chinese", "en": "English", "fr": "French", "pt": "Portuguese",\n    "es": "Spanish", "ja": "Japanese", "tr": "Turkish", "ru": "Russian",\n    "ar": "Arabic", "ko": "Korean", "th": "Thai", "it": "Italian",\n    "de": "German", "vi": "Vietnamese", "ms": "Malay", "id": "Indonesian",\n    "tl": "Filipino", "hi": "Hindi", "zh-hant": "Traditional Chinese",\n    "pl": "Polish", "cs": "Czech", "nl": "Dutch", "km": "Khmer",\n    "my": "Burmese", "fa": "Persian", "gu": "Gujarati", "ur": "Urdu",\n    "te": "Telugu", "mr": "Marathi", "he": "Hebrew", "bn": "Bengali",\n    "ta": "Tamil", "uk": "Ukrainian", "bo": "Tibetan", "kk": "Kazakh",\n    "mn": "Mongolian", "ug": "Uyghur", "yue": "Cantonese",\n}\nSUPPORTED_LANGUAGES = list(LANGUAGE_NAMES)\n\nTOKENIZER = AutoTokenizer.from_pretrained(\n    UPSTREAM_MODEL, revision=UPSTREAM_REVISION, trust_remote_code=True\n)\nMODEL = AutoModelForCausalLM.from_pretrained(\n    UPSTREAM_MODEL,\n    revision=UPSTREAM_REVISION,\n    dtype=torch.bfloat16,\n    device_map={"": 0},\n    trust_remote_code=True,\n    low_cpu_mem_usage=True,\n).eval()\n\ndef translate_exact(texts: list[str], source: str, target: str) -> list[str]:\n    source_key = source.lower()\n    target_key = target.lower()\n    if source_key not in LANGUAGE_NAMES or target_key not in LANGUAGE_NAMES:\n        raise HTTPException(status_code=422, detail=f"unsupported Hy-MT2 language pair: {source} -> {target}")\n    results = []\n    for text in texts:\n        prompt = (\n            f"Translate the following text from {LANGUAGE_NAMES[source_key]} into {LANGUAGE_NAMES[target_key]}. "\n            "Only output the translated result without any additional explanation:\\n"\n            f"{text}"\n        )\n        inputs = TOKENIZER.apply_chat_template(\n            [{"role": "user", "content": prompt}],\n            add_generation_prompt=True,\n            return_tensors="pt",\n            return_dict=True,\n        ).to("cuda")\n        with torch.inference_mode():\n            output = MODEL.generate(\n                **inputs,\n                max_new_tokens=512,\n                do_sample=True,\n                temperature=0.7,\n                top_p=0.6,\n                top_k=20,\n                repetition_penalty=1.05,\n            )\n        generated = output[0][inputs["input_ids"].shape[-1]:]\n        results.append(TOKENIZER.decode(generated, skip_special_tokens=True).strip())\n    return results\n\nTOKEN = os.environ["LA_STUDIO_COLAB_TRANSLATION_TOKEN"]\nMAX_TRANSLATION_SEGMENTS = 128\nMAX_TRANSLATION_CHARS = 50000\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\n\ndef authorize(authorization: str = Header(default="")):\n    if not secrets.compare_digest(authorization, "Bearer " + TOKEN):\n        raise HTTPException(status_code=401, detail="invalid or missing bearer token")\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. Open the notebook for the selected model.",\n        )\n\nclass TranslationSegment(BaseModel):\n    id: str = Field(min_length=1, max_length=128)\n    sourceText: str = Field(min_length=1, max_length=5000)\n\nclass TranslationRequest(BaseModel):\n    model: str\n    source_language: str = Field(min_length=2, max_length=12)\n    target_language: str = Field(min_length=2, max_length=12)\n    segments: list[TranslationSegment]\n\napp = FastAPI(title=f"LA Studio Translation - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(_: None = Depends(authorize)):\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(_: None = Depends(authorize)):\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "translation",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/translations")\ndef translate(request: TranslationRequest, _: None = Depends(authorize)):\n    require_exact_model(request.model)\n    if not request.segments:\n        raise HTTPException(status_code=400, detail="segments must not be empty")\n    texts = [item.sourceText.strip() for item in request.segments]\n    if any(not text for text in texts):\n        raise HTTPException(status_code=400, detail="each segment needs sourceText")\n    if len(texts) > MAX_TRANSLATION_SEGMENTS or sum(map(len, texts)) > MAX_TRANSLATION_CHARS:\n        raise HTTPException(status_code=413, detail="translation request is too large")\n    if not INFERENCE_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="worker is busy; retry shortly")\n    try:\n        translated = translate_exact(\n            texts,\n            request.source_language.strip(),\n            request.target_language.strip(),\n        )\n        if len(translated) != len(request.segments):\n            raise RuntimeError("model returned a different number of translations")\n        return {\n            "patches": [\n                {"id": item.id, "targetText": text.strip(), "state": "translated"}\n                for item, text in zip(request.segments, translated)\n            ]\n        }\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} translation failed: {type(error).__name__}: {str(error)[:300]}",\n        ) from error\n    finally:\n        INFERENCE_SLOTS.release()' + '\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'hy-mt2-1.8b'
# LA Studio worker launch contract: launch-2026-07-30.1
import json
import os
import queue
import re
import secrets
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'translation'
MODEL_ID = 'hy-mt2-1.8b'
PORT = 3943
TOKEN_ENV = 'LA_STUDIO_COLAB_TRANSLATION_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_TRANSLATION_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_TRANSLATION_MODEL'
WORKER_LOG = Path('/content/la_studio_translation_worker.log')
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


if port_is_occupied(PORT):
    raise RuntimeError(
        f"Port {PORT} is already occupied by an earlier Colab worker. "
        "Use Runtime > Disconnect and delete runtime, then Run all once for this exact model."
    )

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", 'la_studio_translation_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    print(f"Starting exact CUDA {CAPABILITY_LABEL} worker; initial model load can take several minutes.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower() == "cuda"
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print("Exact CUDA worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print("Waiting for the exact CUDA model…", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The exact-model {CAPABILITY_LABEL} worker did not become CUDA-ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
